# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [2]:
%pip install gradio

   ---------------------------------------- 0.0/24.2 MB ? eta -:--:--
   -- ------------------------------------- 1.3/24.2 MB 9.3 MB/s eta 0:00:03
   ---------- ----------------------------- 6.3/24.2 MB 18.8 MB/s eta 0:00:01
   -------------------- ------------------- 12.3/24.2 MB 22.4 MB/s eta 0:00:01
   -------------------------- ------------- 16.0/24.2 MB 21.0 MB/s eta 0:00:01
   ------------------------------------ --- 21.8/24.2 MB 24.1 MB/s eta 0:00:01
   ---------------------------------------- 24.2/24.2 MB 22.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/553.3 kB ? eta -:--:--
   --------------------------------------- 553.3/553.3 kB 12.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   -------------------------------- ------- 10.0/12.3 MB 47.8 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 41.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --------


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: C:\Users\john louie polo\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [4]:
import gradio as gr
from openai import OpenAI
import os
import json
import time

# --- 1. Setup Environment ---
# (Using the Gemini configuration from your previous exercises)
os.environ["GEMINI_API_KEY"] = "your-gemini-key-here" # Replace with your key

client = OpenAI(
    api_key=os.environ.get("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# --- 2. Define a Custom Tool (From Day 4) ---
# This tool simulates checking a database for your motor parts POS system
def check_inventory(part_name):
    print(f"[SYSTEM] Database tool called for: {part_name}")
    # Simulated database
    mock_db = {"spark plug": 45, "brake pad": 12, "oil filter": 0, "drive belt": 5}
    stock = mock_db.get(part_name.lower(), "Part not found in inventory")
    return f"Inventory status for {part_name}: {stock}"

# The JSON schema required to tell the LLM how to use our tool
inventory_tool = {
    "name": "check_inventory",
    "description": "Check the current inventory stock level for a specific motor part.",
    "parameters": {
        "type": "object",
        "properties": {
            "part_name": {"type": "string", "description": "The name of the motor part, e.g., spark plug"}
        },
        "required": ["part_name"]
    }
}
tools = [{"type": "function", "function": inventory_tool}]

def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "check_inventory":
            arguments = json.loads(tool_call.function.arguments)
            part = arguments.get('part_name')
            result = check_inventory(part)
            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })
    return responses

# --- 3. The Chat Callback Logic (From Day 3 & 5) ---
def chat_logic(message, history, system_prompt, model_choice):
    # 1. Format the history array for the API
    messages = [{"role": "system", "content": system_prompt}]
    for h in history:
        messages.append({"role": h["role"], "content": h["content"]})
    
    messages.append({"role": "user", "content": message})
    
    # 2. Initial API call (Checking if the AI wants to use a tool)
    response = client.chat.completions.create(
        model=model_choice,
        messages=messages,
        tools=tools
    )
    
    # 3. Handle tools if the AI requested them
    while response.choices[0].finish_reason == "tool_calls":
        ai_msg = response.choices[0].message
        messages.append(ai_msg)
        
        # Execute the Python code and append the result
        tool_responses = handle_tool_calls(ai_msg)
        messages.extend(tool_responses)
        
        # Send the tool result back to the AI
        response = client.chat.completions.create(
            model=model_choice,
            messages=messages,
            tools=tools
        )
        
    # 4. Stream the final response to the Gradio UI
    final_text = response.choices[0].message.content
    partial_message = ""
    for char in final_text:
        partial_message += char
        time.sleep(0.005) # Creates a smooth typing effect
        yield partial_message

# --- 4. The Gradio Interface (From Day 2 & 5) ---
# --- 4. The Gradio Interface (Updated for Gradio 6.0) ---
# Removed the theme argument from here
with gr.Blocks() as demo:
    gr.Markdown("# 🛠️ Technical Q&A Prototype with Tools")
    gr.Markdown("Test this by asking: *'Do we have any spark plugs or oil filters in stock?'*")
    
    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=["gemini-1.5-flash", "gemini-1.5-pro", "llama3.2"], 
            value="gemini-1.5-flash", 
            label="Select Model"
        )
        system_prompt = gr.Textbox(
            value="You are a senior technical advisor specializing in Node.js, Supabase, and regional management platforms like BiCoRSE. Give concise, highly technical answers.",
            label="System Expertise (System Prompt)"
        )
        
    # Removed the type="messages" argument here
    chat_ui = gr.ChatInterface(
        fn=chat_logic,
        additional_inputs=[system_prompt, model_dropdown]
    )

# Moved the theme argument down here to the launch method
demo.launch(inbrowser=True, theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "C:\Users\john louie polo\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "C:\Users\john louie polo\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "C:\Users\john louie polo\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\blocks.py", line 2157, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "C:\Users\john louie polo\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\blocks.py", line 1646, in call_function
    prediction = await utils.as